# Why baby cry - Kaggle version

Audio classification of infant cries using [fastaudio](https://github.com/fastaudio/fastaudio) + fastai.

This notebook is prepared to run on **[Kaggle](https://www.kaggle.com)** using the **donateacry_corpus** dataset.

## How to attach the dataset

1. Open this notebook on Kaggle.
2. In the right panel click **Add Input** (or **Add Data**) and search for the dataset named **donateacry_corpus**.
3. Add it. Kaggle mounts every dataset read-only under `/kaggle/input/<dataset-slug>/`.
4. Make sure a **GPU** accelerator is enabled (Settings -> Accelerator) and **Internet** is ON so the `pip install` cell can fetch fastaudio.

The dataset keeps the same class subfolders:

```
donateacry_corpus/
  belly_pain/
  burping/
  discomfort/
  hungry/
  tired/
```

Each audio file already lives inside its class folder, so the label is simply the name of the parent directory.

## 1. Install fastaudio

In [ ]:
# Requires Internet ON (Settings -> Internet).
!pip install -q git+https://github.com/fastaudio/fastaudio.git

In [ ]:
from fastaudio.core.all import *
from fastaudio.augment.all import *
from fastai.vision.all import *
import pandas as pd

## 2. Locate the dataset

On Kaggle the added datasets are mounted under `/kaggle/input/`. We search for the
`donateacry_corpus` folder automatically so the notebook keeps working even if the
dataset slug differs slightly (e.g. `donateacry-corpus`).

In [ ]:
from pathlib import Path
from collections import Counter

INPUT_ROOT = Path('/kaggle/input')

def find_corpus_root(input_root=INPUT_ROOT):
    """Return the folder that contains the class subfolders of donateacry_corpus.

    Handles both layouts:
      /kaggle/input/<slug>/donateacry_corpus/<class>/*.wav
      /kaggle/input/<slug>/<class>/*.wav
    """
    expected_classes = {'belly_pain', 'burping', 'discomfort', 'hungry', 'tired'}

    # 1) A directory literally named donateacry_corpus that has class subfolders.
    for p in input_root.rglob('donateacry_corpus'):
        if p.is_dir() and any((p / c).is_dir() for c in expected_classes):
            return p

    # 2) Any directory that directly contains the expected class subfolders.
    for p in input_root.rglob('*'):
        if p.is_dir() and expected_classes.issubset({c.name for c in p.iterdir() if c.is_dir()}):
            return p

    raise FileNotFoundError(
        'Could not find donateacry_corpus under /kaggle/input. '
        'Add the dataset via "Add Input" in the right panel.'
    )

path_audio = find_corpus_root()
print('Dataset root:', path_audio)

In [ ]:
# Inspect how many audio files exist per class (one subfolder per class).
wav_files = list(path_audio.rglob('*.wav'))
counts = Counter(f.parent.name for f in wav_files)

print(f'Total .wav files: {len(wav_files)}')
print('Files per class:')
for label, n in sorted(counts.items()):
    print(f'  {label:12s} {n}')

## 3. Build the audio pipeline

Because `/kaggle/input` is read-only, we point the spectrogram cache at the writable
`/kaggle/working` folder.

In [ ]:
# In the donateacry_corpus each file lives inside its class subfolder,
# so the label is simply the name of the parent directory.
def label_func(fname):
    return Path(fname).parent.name

In [ ]:
# /kaggle/input is read-only, so keep any generated cache in /kaggle/working.
path_audio_cached = Path('/kaggle/working/audio_cached')
path_audio_cached.mkdir(parents=True, exist_ok=True)

In [ ]:
audioBlock = AudioBlock.from_folder(path_audio, sample_rate=2000)
crop_time  = ResizeSignal(2000)
silence    = RemoveSilence()

In [ ]:
sg_cfg = AudioConfig.BasicSpectrogram(n_fft=2000, hop_length=155)
a2sg   = AudioToSpec.from_cfg(sg_cfg)

In [ ]:
aud_digit = DataBlock(blocks=(AudioBlock, CategoryBlock),
                 get_items=get_audio_files,
                 splitter=RandomSplitter(),
                 item_tfms=[silence, Normalize(), crop_time, a2sg],
                 get_y=label_func)

In [ ]:
# get_audio_files searches recursively, so it picks up every .wav
# inside each class subfolder of the corpus.
dbunch = aud_digit.dataloaders(path_audio, bs=64)

In [ ]:
dbunch.show_batch(max_n=6)

## 4. Train the model

In [ ]:
learn = cnn_learner(dbunch,
            resnet34,
            config=cnn_config(n_in=1),  # <- Only audio specific modification here
            loss_fn=CrossEntropyLossFlat,
            metrics=[accuracy])

In [ ]:
lr = learn.lr_find()
lr

In [ ]:
# Use the suggested learning rate from lr_find (falls back to a sane default).
base_lr = getattr(lr, 'valley', None) or getattr(lr, 'lr_min', None) or 1e-2
learn.fine_tune(10, base_lr=base_lr)

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(6, 6))

## 5. Export the trained model

Save into `/kaggle/working` so it becomes a downloadable output of the notebook.

In [ ]:
export_path = Path('/kaggle/working/export.pkl')
learn.export(export_path)
print('Model exported to', export_path)

## 6. Test the exported model

Reload the saved learner and run a prediction on one file from the dataset.
Replace `sample` with your own `.wav` path to test a specific audio.

In [ ]:
loaded_model = load_learner('/kaggle/working/export.pkl')

# Pick any file from the corpus as a quick smoke test.
sample = wav_files[0]
print('Predicting for:', sample)

pred_class, pred_idx, probs = loaded_model.predict(sample)
print('Predicted class:', pred_class)
print('Probabilities  :', dict(zip(loaded_model.dls.vocab, map(float, probs))))